# Generation of a simple DFN with PorePy and Transfer to OGS

This is work in progress

In [1]:
import numpy as np
import porepy as pp
import ogstools as ot
import ogs as ogs

/home/mok/.venv/lib/python3.12/site-packages/porepy/numerics/nonlinear/nonlinear_solvers.py:14: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import trange  # type: ignore


# Setting up the domain and generating a random set of circular fractures

In [2]:
mins = np.array([0.,0.,0.])
maxs = np.array([10.,10.,10.])

In [3]:
bounding_box = {'xmin': mins[0], 'xmax': maxs[0], 'ymin': mins[1], 'ymax': maxs[1], 'zmin': mins[2], 'zmax': maxs[2]}
domain = pp.Domain(bounding_box=bounding_box)
domain

pp.Domain(bounding_box={'xmin': np.float64(0.0), 'xmax': np.float64(10.0), 'ymin': np.float64(0.0), 'ymax': np.float64(10.0), 'zmin': np.float64(0.0), 'zmax': np.float64(10.0)})

In [4]:
nfracs = 8
r_range = np.array([3,9])
f_i = np.array([])
for i in range(nfracs):
    center = np.random.rand(3) * (maxs - mins) + mins
    major_axis = np.random.rand() * (r_range[1] - r_range[0]) + r_range[0]
    minor_axis = major_axis.copy() #circular
    major_axis_angle = 0. #for circular
    strike_angle = np.random.rand() * np.pi - np.pi/2
    dip_angle = np.random.rand() * np.pi - np.pi/2
    f_i = np.append(f_i,pp.create_elliptic_fracture(center, major_axis, minor_axis, major_axis_angle, strike_angle, dip_angle))

In [5]:
network = pp.create_fracture_network(fractures=f_i,domain=domain)
network

Three-dimensional fracture network with 8 plane fractures.
The domain is a cuboid with bounding box: {'xmin': np.float64(0.0), 'xmax': np.float64(10.0), 'ymin': np.float64(0.0), 'ymax': np.float64(10.0), 'zmin': np.float64(0.0), 'zmax': np.float64(10.0)}.

## Meshing ... 

In [6]:
mesh_args = {'cell_size_boundary': 1.0, 'cell_size_fracture': 0.5, 'cell_size_min': 0.1}
mdg = pp.create_mdg("simplex", mesh_args, network)

In [7]:
#Removal of 3D not needed really
mdg2d = mdg.copy()
for sd in mdg2d.subdomains():
    if sd.dim == 3:
        mdg2d.remove_subdomain(sd)
mdg2d

Mixed-dimensional grid containing 50 grids and 112 interfaces.
Maximum dimension present: 2 
Minimum dimension present: 0 
8 grids of dimension 2 with in total 11653 cells
35 grids of dimension 1 with in total 313 cells
7 grids of dimension 0 with in total 7 cells
70 interfaces between grids of dimension 2 and 1 with in total 1252 mortar cells.
42 interfaces between grids of dimension 1 and 0 with in total 42 mortar cells.

In [8]:
#pp.plot_grid(mdg2d, figsize=(12,12), plot_2d=False)

## Export to VTU and import in OGS. Setting up Material IDs

In [9]:
pp.Exporter(mdg2d, 'mixed_dimensional_grid').write_vtu()

In [10]:
DFN_2D = ot.Mesh('mixed_dimensional_grid_constant_2.vtu')
DFN_2D

Mesh (0x7b0924a020e0)
  N Cells:    11653
  N Points:   6818
  X Bounds:   0.000e+00, 1.000e+01
  Y Bounds:   0.000e+00, 1.000e+01
  Z Bounds:   0.000e+00, 1.000e+01
  N Arrays:   6

In [11]:
DFN_2D['MaterialIDs'] = DFN_2D['subdomain_id'] - DFN_2D['subdomain_id'].min()

In [12]:
fig = DFN_2D.plot('MaterialIDs',show_edges=True)

Widget(value='<iframe src="http://localhost:40135/index.html?ui=P_0x7b09258e3f20_0&reconnect=auto" class="pyvi…

## Generating boundaries for OGS

In [13]:
ogs.cli.ExtractBoundary(i = 'mixed_dimensional_grid_constant_2.vtu', o='boundaries.vtu')

[2025-03-29 18:10:10.979] [ogs] [info] Mesh read: 6818 nodes, 11653 elements.
[2025-03-29 18:10:10.985] [ogs] [info] 6 property vectors copied, 0 vectors skipped.
[2025-03-29 18:10:10.985] [ogs] [info] Created surface mesh: 1913 nodes, 1913 elements.


0

In [14]:
tol = 1e-3
ogs.cli.removeMeshElements(i='boundaries.vtu',o='xmax.vtu',**{"x-max": maxs[0]-tol})
ogs.cli.removeMeshElements(i='boundaries.vtu',o='xmin.vtu',**{"x-min": mins[0]+tol})
#
ogs.cli.removeMeshElements(i='boundaries.vtu',o='ymax.vtu',**{"y-max": maxs[1]-tol})
ogs.cli.removeMeshElements(i='boundaries.vtu',o='ymin.vtu',**{"y-min": mins[1]+tol})
#
ogs.cli.removeMeshElements(i='boundaries.vtu',o='zmax.vtu',**{"z-max": maxs[2]-tol})
ogs.cli.removeMeshElements(i='boundaries.vtu',o='zmin.vtu',**{"z-min": mins[2]+tol})

[2025-03-29 18:10:11.045] [ogs] [info] Mesh read: 1913 nodes, 1913 elements.
[2025-03-29 18:10:11.045] [ogs] [info] Bounding box of "boundaries" is
x = [0.000000,10.000000]
y = [0.000000,10.000000]
z = [0.000000,10.000000]
[2025-03-29 18:10:11.045] [ogs] [info] 1841 elements found.
[2025-03-29 18:10:11.045] [ogs] [info] Removing total 1841 elements...
[2025-03-29 18:10:11.045] [ogs] [info] 72 elements remain in mesh.
[2025-03-29 18:10:11.046] [ogs] [info] Removing total 1832 nodes...
[2025-03-29 18:10:11.076] [ogs] [info] Mesh read: 1913 nodes, 1913 elements.
[2025-03-29 18:10:11.076] [ogs] [info] Bounding box of "boundaries" is
x = [0.000000,10.000000]
y = [0.000000,10.000000]
z = [0.000000,10.000000]
[2025-03-29 18:10:11.076] [ogs] [info] 1865 elements found.
[2025-03-29 18:10:11.076] [ogs] [info] Removing total 1865 elements...
[2025-03-29 18:10:11.076] [ogs] [info] 48 elements remain in mesh.
[2025-03-29 18:10:11.077] [ogs] [info] Removing total 1862 nodes...
[2025-03-29 18:10:11.1

0